In [0]:
%sql
CREATE VOLUME IF NOT EXISTS uc.urise.urise_etc;



In [0]:
# Bypass CSV - langsung baca dari tabel existing
df_may = spark.table("uc.urise.urise_transaction_may")
df_june = spark.table("uc.urise.urise_transaction_june")

df = df_may.unionByName(df_june)
print(f"Total rows (May + June): {df.count()}")

df.write.mode("overwrite").saveAsTable('uc.urise.rom_bronze')

In [0]:
# Schema uc.urise sudah ada, skip create schema

In [0]:
df_rom_bronze = spark.table("uc.urise.rom_bronze")

In [0]:
import pandas as pd

df = df_rom_bronze.toPandas()

# Konversi room_in menjadi datetime
df["room_in"] = pd.to_datetime(df["room_in"])

# Ambil jam
df["hour"] = df["room_in"].dt.hour

# Tentukan tanggal operasional
# Jika sebelum jam 06:00 maka dianggap hari sebelumnya
# operational_datetime = df["room_in"].where(
#     df["hour"] >= 6,
#     df["room_in"] - pd.Timedelta(days=1)
# )
operational_datetime = df["room_in"] - pd.Timedelta(hours=6)

df['date'] = df['room_in'].dt.date
df["op_date"] = operational_datetime.dt.date
df['op_hour'] = operational_datetime.dt.hour
df['op_datetime'] = operational_datetime.dt.floor('h')

# Tentukan shift
df["shift"] = df["hour"].apply(
    lambda h: "DS" if 6 <= h < 18 else "NS"
)


# ============================================
dfs = spark.createDataFrame(df)
dfs.write.mode("overwrite").saveAsTable('uc.urise.rom_silver')

In [0]:
# Schema uc.urise sudah ada, skip create schema

In [0]:
df_rom_silver = spark.table("uc.urise.rom_silver")

In [0]:
import pandas as pd

df = df_rom_silver.toPandas()

# Pastikan tonase bertipe numerik
df["tonase"] = pd.to_numeric(df["tonase"], errors="coerce").fillna(0)

# Agregasi
gold = (
    df.groupby(
        ["op_date", "op_hour", "op_datetime", "date", "hour", "shift", "master_location", "kontraktor"],
        as_index=False
    )
    .agg(
        jumlah_transaksi=("id", "count"),
        jumlah_unit_aktif=("no_lambung", "nunique"),
        tonase=("tonase", "sum")
    )
)

# (Opsional) Urutkan hasil
gold = gold.sort_values(
    ["op_date", "op_hour", "master_location", "kontraktor"]
)


# # Simpan hasil
# gold.to_csv(output_file, index=False)

# print(f"Agregasi selesai. Hasil disimpan ke {output_file}")


dfg = spark.createDataFrame(gold)
dfg.write.mode("overwrite").saveAsTable('uc.urise.rom')

In [0]:
df_rom_gold = spark.table("uc.urise.rom")
display(df_rom_gold)